Doing the required imports

In [1]:
import datetime
import os
import shutil
import sys

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../data"))

import dataset

os.chdir("..")

Define the constants

In [2]:
SPLITS = 7
IMAGE_SIZE = 640

Creating the data directories

In [3]:
dataset.create_dataset_directories()

Get the data folds

In [4]:
(
    labels,
    classes,
    kfolds,
    labels_df,
    labels_df_with_counts,
    cls_idx,
    folds_df,
    valid_labels_df,
) = dataset.get_data_folds()

Create the yaml directories

In [5]:
images, save_path, ds_yamls = dataset.create_yml_directories(folds_df, classes)

Copy validation data

In [6]:
dataset.copy_validation_data(images, labels, valid_labels_df, save_path)

Fucntion to copy images

In [7]:
def copy_images_and_labels(
    images, labels, folds_df, save_path, k
):
    for image, label in zip(images, labels):
        if image.stem not in folds_df.index:
            continue
        split = f"split_{k + 1}"
        k_split = folds_df.loc[image.stem][split]
        # Destination directory
        img_to_path = save_path / split / k_split / "images"
        lbl_to_path = save_path / split / k_split / "labels"
        image_name = f"{image.stem}{image.suffix}"
        label_name = f"{label.stem}{label.suffix}"
        img = dataset.load_image(image)
        img.save(img_to_path / image_name)
        shutil.copy(label, lbl_to_path / label_name)

Model training

In [8]:
from ultralytics import YOLO

In [9]:
def train_models(ds_yamls, images, labels, folds_df, save_path):
    results = {}

    # Define your additional arguments here
    folds = SPLITS
    batch = 16
    current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    project = f"runs/train/{folds}fold/{current_time}"
    epochs = 40

    for k in range(folds):
        copy_images_and_labels(images, labels, folds_df, save_path, k)
        dataset_yaml = ds_yamls[k]
        model = YOLO("yolo11s.pt", task="detect")
        model.train(
            data=dataset_yaml,
            epochs=epochs,
            batch=batch,
            imgsz=IMAGE_SIZE,
            project=project,
            patience=10,
            seed=0,
            optimizer="AdamW",
            lr0=0.0005,
            momentum=0.9,
            flipud=0.5,
            crop_fraction=0.9,
            mixup=0.1,
            cls=2,
        )
        results[k] = model.metrics

        model.val(
            data=dataset_yaml,
            project=f"{project}/validation",
            imgsz=IMAGE_SIZE,
        )

        shutil.rmtree(f"{save_path}/split_{k + 1}")

    return project

In [10]:
train_models(ds_yamls, images, labels, folds_df, save_path)

New https://pypi.org/project/ultralytics/8.3.159 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.39 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 7974MiB)
engine/trainer: task=detect, mode=train, model=yolo11s.pt, data=data/dataset_cross_validation/2025-06-26_7-Fold_Cross-val/split_1/split_1_dataset.yaml, epochs=40, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=runs/train/7fold/2025-06-26 17:25, name=train, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visuali

train: Scanning /home/mirikwa/Projects/zindi/Amini-Cocoa-Contamination-Challenge-Solution/data/dataset_cross_validation/

train: New cache created: /home/mirikwa/Projects/zindi/Amini-Cocoa-Contamination-Challenge-Solution/data/dataset_cross_validation/2025-06-26_7-Fold_Cross-val/split_1/train/labels.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/home/mirikwa/.local/lib/python3.10/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/home/mirikwa/.local/lib/python3.10/site-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /home/mirikwa/Projects/zindi/Amini-Cocoa-Contamination-Challenge-Solution/data/dataset_cross_validation/20


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 25/25 [00:05<


                   all        789       1398      0.804      0.716      0.791      0.541
           anthracnose        226        322      0.801      0.742      0.812      0.563
                 cssvd        321        473      0.834      0.662      0.746      0.511
               healthy        242        603      0.778      0.745      0.814       0.55
Speed: 0.5ms preprocess, 3.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/train/7fold/2025-06-26 17:25/train7
Ultralytics 8.3.39 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 7974MiB)
YOLO11s summary (fused): 238 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs


val: Scanning /home/mirikwa/Projects/zindi/Amini-Cocoa-Contamination-Challenge-Solution/data/dataset_cross_validation/20
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 50/50 [00:09<


                   all        789       1398        0.8       0.72      0.791      0.541
           anthracnose        226        322      0.798      0.748      0.812      0.564
                 cssvd        321        473       0.83      0.662      0.745       0.51
               healthy        242        603      0.772      0.751      0.815       0.55
Speed: 0.5ms preprocess, 5.6ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs/train/7fold/2025-06-26 17:25/validation/train7


'runs/train/7fold/2025-06-26 17:25'

Doing the inference

In [11]:
import inference

In [12]:
inference.do_prediction(
    "runs/train/7fold/2025-05-07 01:14",
    ["data/dataset/images/test"],
    "output/Submission151.csv",
    confidence=0.001,
    iou_threshold=0.5,
    max_detection=int(300),
)

100%|███████████████████████████████████████████████████████████████████████████████| 1626/1626 [08:44<00:00,  3.10it/s]
